In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import re
import random

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mateuscpinheiro/arquivos-bblia/bibliaAnalisada.csv
/kaggle/input/datasets/bradystephenson/bibledata/BibleData-PlaceVerse.csv
/kaggle/input/datasets/bradystephenson/bibledata/BibleData-Event.csv
/kaggle/input/datasets/bradystephenson/bibledata/AlamoPolyglot.csv
/kaggle/input/datasets/bradystephenson/bibledata/HitchcocksBibleNamesDictionary.csv
/kaggle/input/datasets/bradystephenson/bibledata/LICENSE
/kaggle/input/datasets/bradystephenson/bibledata/BibleData-Reference.csv
/kaggle/input/datasets/bradystephenson/bibledata/BibleData-Place.csv
/kaggle/input/datasets/bradystephenson/bibledata/BibleData-Epoch.csv
/kaggle/input/datasets/bradystephenson/bibledata/HebrewStrongs.csv
/kaggle/input/datasets/bradystephenson/bibledata/BibleData-PersonRelationship.csv
/kaggle/input/datasets/bradystephenson/bibledata/README.md
/kaggle/input/datasets/bradystephenson/bibledata/BibleData-Commandments.csv
/kaggle/input/datasets/bradystephenson/bibledata/NavesTopicalDictionary.csv
/kag

In [2]:
siglas = { 
    'Ageu': 'Ag', 'Amos': 'Am', 'Apocalipse': 'Ap', 'Atos': 'At', 
    'Cantico dos Canticos': 'Ct','Colossenses': 'Cl', 
    'Daniel': 'Dn', 'Deuteronomio': 'Dt', 
    'Eclesiastes': 'Ec', 'Efesios': 'Ef','Esdras': 'Es', 'Ester': 'Et', 'Exodo': 'Ex', 'Ezequiel': 'Ez', 
    'Filemom': 'Flm', 'Filipenses': 'Fl',
    'Galatas': 'Gl', 'Genesis': 'Gn', 
    'Habacuque':'Hc', 'Hebreus': 'Hb', 
    'I Corintios': '1Co', 'I Cronicas': '1Cr', 'I Joao': '1Jo', 'I Pedro': '1Pe', 'I Reis': '1Re', 'I Samuel': '1Sm',
    'I Tessalonicenses': '1Ts', 'I Timoteo': '1Tm', 
    'II Corintios': '2Co', 'II Cronicas': '2Cr', 'II Joao': '2Jo', 'II Pedro': '2Pe', 'II Reis': '2Re', 'II Samuel': '2Sm',
    'II Tessalonicenses': '2Ts', 'II Timoteo': '2Tm',
    'III Joao': '3Jo',
    'Isaias': 'Is',
    'Jeremias': 'Jr', 'Jo': 'Jó', 'Joao': 'Jo', 'Joel': 'Jl', 'Jonas': 'Jon', 'Josue': 'Js', 'Judas': 'Jd', 'Juizes': 'Jz', 
    'Lamentacoes Jeremias': 'Lm', 'Levitico': 'Lv', 'Lucas': 'Lc', 
    'Malaquias': 'Ml', 'Marcos': 'Mc', 'Mateus': 'Mt', 'Miqueias': 'Mq', 
    'Naum': 'Na', 'Neemias': 'Ne', 'Numeros': 'Nm',
    'Obadias': 'Ob', 'Oseias': 'Os', 
    'Proverbios': 'Pv', 
    'Romanos': 'Rm', 'Rute': 'Rt', 
    'Salmos': 'Sl', 'Sofonias': 'Sf', 
    'Tiago': 'Tg', 'Tito': 'Tt', 
    'Zacarias': 'Zc'
    }
print(len(siglas))



66


In [3]:
biblia_sentimentos = pd.read_csv("/kaggle/input/datasets/paulogladson/biblia-sentiment/biblia-analise-sentimentos.csv",sep="|")

biblia_sentimentos['ref'] = biblia_sentimentos.apply(lambda x: str(siglas[x.livro]) + ' ' + str(x.capitulo) + ':' + str(x.versiculo), axis=1)

# Função que limpa os sub-tokens colando-os na palavra anterior
def limpar_tokens_avancado(texto):
    if isinstance(texto, str):
        # Limpa os ## e a pontuação antes deles
        texto_limpo = re.sub(r"[,\s'\"]*##", '', texto)
        # Tira vírgulas soltas no final
        return texto_limpo.strip(', ')
    return texto

biblia_sentimentos['Pessoas'] = biblia_sentimentos['Pessoas'].apply(limpar_tokens_avancado)
biblia_sentimentos['Locais'] = biblia_sentimentos['Locais'].apply(limpar_tokens_avancado)
biblia_sentimentos['Organizacoes'] = biblia_sentimentos['Organizacoes'].apply(limpar_tokens_avancado)
biblia_sentimentos['Miscelanea'] = biblia_sentimentos['Miscelanea'].apply(limpar_tokens_avancado)

biblia_sentimentos.drop_duplicates(subset='ref', inplace=True)
biblia_sentimentos.set_index('ref',inplace=True)
biblia_sentimentos.drop(columns = ['Unnamed: 0', 'versao','testamento','livro','capitulo','versiculo'], inplace=True)

print(biblia_sentimentos.columns)
print(biblia_sentimentos.shape)
biblia_sentimentos.sample(15)

Index(['posicao', 'texto', 'Pessoas', 'Locais', 'Organizacoes', 'Miscelanea',
       'label', 'score'],
      dtype='object')
(31482, 8)


,posicao,texto,Pessoas,Locais,Organizacoes,Miscelanea,label,score
ref,,,,,,,,
2Cr 6:15,14,"Que guardaste ao teu servo Davi, meu pai, o qu...",Davi,NaN,NaN,NaN,positive,0.492286
2Cr 30:18,14,"Porque uma multidão do povo, muitos de Efraim ...",fraimsséssacarbulom,"E, Mana, Is, Ze",NaN,NaN,negative,0.638841
Jz 8:9,7,"Por isso também falou aos homens de Penuel, di...",uel,Pen,NaN,NaN,negative,0.829862
Jo 1:46,4,Disse-lhe Natanael: Pode vir alguma coisa boa ...,"Natanael, Nazaré, Filipe",NaN,NaN,NaN,positive,0.561558
Ez 25:12,26,Assim diz o Senhor DEUS: Porquanto Edom se hou...,"m, Judá",Edo,NaN,NaN,negative,0.879924
1Re 4:18,11,"Simei, filho de Elá, em Benjamim:","Simei, Elá",Benjamim,NaN,NaN,positive,0.611316
Jr 25:2,24,A qual anunciou o profeta Jeremias a todo o po...,Jeremias,"Judá, Jerusalém",NaN,NaN,positive,0.598809
Sl 19:11,19,Também por eles é admoestado o teu servo,NaN,NaN,NaN,NaN,positive,0.608073
At 28:27,5,"Porque o coração deste povo se endureceu, e co...",NaN,NaN,NaN,NaN,negative,0.769249


In [4]:
biblia_analisada = pd.read_csv("/kaggle/input/datasets/mateuscpinheiro/arquivos-bblia/bibliaAnalisada.csv",sep="|",encoding='utf-8')
biblia_analisada['ref'] = biblia_analisada.apply(lambda x: str(siglas[x.livro]) + ' ' + str(x.capitulo) + ':' + str(x.versiculo), axis=1)

mapa_versoes = {
    'Almeida Corrigida e Revisada Fiel': 'ACF',
    'Almeida Revisada Imprensa Bíblica': 'ARIB',
    'Almeida Revista e Atualizada': 'ARA',
    'Não Identificada': 'NI',
    'Nova Versão Internacional': 'NVI',
    'Sociedade Bíblica Britânica': 'SBB'
}

# Criamos uma coluna temporária com a sigla no seu DataFrame original
biblia_analisada['sigla'] = biblia_analisada['versao'].map(mapa_versoes)

# Pivotiando o dataframe para separar texto e posicao por sigla
biblia_analisada_pivot = biblia_analisada.pivot(
    index=['ref', 'testamento', 'livro', 'capitulo', 'versiculo'], 
    columns='sigla', 
    values=['texto']
)

# O pivot gera colunas em multi-índice (ex: ('texto', 'ACF')). Vamos "achatar" isso:
biblia_analisada_pivot.columns = [f"{col[0]}_{col[1]}" for col in biblia_analisada_pivot.columns]

# Resetamos o índice para trazer testamento, livro, capitulo e versiculo de volta como colunas normais
biblia_analisada_final = biblia_analisada_pivot.reset_index()

biblia_analisada_final.drop_duplicates(subset='ref', inplace=True)
biblia_analisada = biblia_analisada_final.set_index('ref')

print(biblia_analisada.columns)
print(biblia_analisada.shape)

biblia_analisada.sample(15)

Index(['testamento', 'livro', 'capitulo', 'versiculo', 'texto_ACF',
       'texto_ARA', 'texto_ARIB', 'texto_NI', 'texto_NVI', 'texto_SBB'],
      dtype='object')
(31482, 10)


,testamento,livro,capitulo,versiculo,texto_ACF,texto_ARA,texto_ARIB,texto_NI,texto_NVI,texto_SBB
ref,,,,,,,,,,
Jz 21:10,Antigo Testamento,Juizes,21,10,Então a assembléia enviou para lá doze mil hom...,Pelo que a congregação enviou para lá doze mil...,Pelo que a congregação enviou para lá doze mil...,Pelo que a congregação enviou para lá doze mil...,Então a comunidade enviou doze mil homens de g...,A congregação enviou para lá doze mil homens d...
Lc 2:1,Novo Testamento,Lucas,2,1,E aconteceu naqueles dias que saiu um decreto ...,Naqueles dias saiu um decreto da parte de Césa...,Naqueles dias saiu um decreto da parte de Césa...,Naqueles dias saiu um decreto da parte de Césa...,Naqueles dias César Augusto publicou um decret...,Naqueles dias foi expedido um decreto de César...
Jr 3:18,Antigo Testamento,Jeremias,3,18,Naqueles dias andará a casa de Judá com a casa...,Naqueles dias andará a casa de Judá com a casa...,Naqueles dias andará a casa de Judá com a casa...,Naqueles dias andará a casa de Judá com a casa...,Naqueles dias a comunidade de Judá caminhará c...,Naqueles dias a casa de Judá andará com a casa...
Nm 32:34,Antigo Testamento,Numeros,32,34,"E os filhos de Gade edificaram a Dibom, e Atar...","Os filhos de Gade, pois, edificaram a Dibom, A...","Os filhos de Gade, pois, edificaram a Dibom, A...","Os filhos de Gade, pois, edificaram a Dibom, A...","A tribo de Gade construiu Dibom, Atarote, Aroer,","Os filhos de Gade reedificaram a Dibom, a Atro..."
2Re 20:2,Antigo Testamento,II Reis,20,2,"Então virou o rosto para a parede, e orou ao S...","Então o rei virou o rosto para a parede, e oro...","Então o rei virou o rosto para a parede, e oro...","Então o rei virou o rosto para a parede, e oro...",Ezequias virou o rosto para a parede e orou ao...,"O rei virou o rosto para a parede, e orou a Je..."
Ne 5:2,Antigo Testamento,Neemias,5,2,"Porque havia quem dizia: Nós, nossos filhos e ...","Pois havia alguns que diziam: Nós, nossos filh...","Pois havia alguns que diziam: Nós, nossos filh...","Pois havia alguns que diziam: Nós, nossos filh...","Alguns diziam: ""Nós e nossos filhos e filhas s...","Pois havia quem dissesse: Nós, nossos filhos e..."
1Co 15:10,Novo Testamento,I Corintios,15,10,Mas pela graça de Deus sou o que sou,"Mas pela graça de Deus sou o que sou, e a sua ...",Mas pela graça de Deus sou o que sou,"Mas pela graça de Deus sou o que sou, e a sua ...","Mas, pela graça de Deus, sou o que sou, e sua ...","mas pela graça de Deus sou o que sou, e a sua ..."
Is 48:16,Antigo Testamento,Isaias,48,16,"Chegai-vos a mim, ouvi isto: Não falei em segr...","Chegai-vos a mim, ouvi isto: Não falei em segr...","Chegai-vos a mim, ouvi isto: Não falei em segr...","Chegai-vos a mim, ouvi isto: Não falei em segr...","""Aproximem-se de mim e escutem isto: ""Desde o ...","Chegai-vos perto de mim, ouvi isto"
1Cr 19:5,Antigo Testamento,I Cronicas,19,5,"E foram-se, e avisaram a Davi acerca daqueles ...",Então foram alguns e avisaram a Davi acerca de...,Então foram alguns e avisaram a Davi acerca de...,Então foram alguns e avisaram a Davi acerca de...,"Quando Davi soube disso, enviou mensageiros ao...",Então foram alguns e contaram a Davi como fora...


In [5]:
# adicionar king James 
# strong em hebreu e grego
# https://www.kaggle.com/datasets/bradystephenson/bibledata?select=AlamoPolyglot.csv

In [6]:
biblia_almeida = pd.read_csv("/kaggle/input/datasets/luisguilhermeribeiro/bibliaptbr/biblia_almeida_completa.csv",encoding='utf-8')

biblia_almeida['ref'] = biblia_almeida.apply(lambda x: str(siglas[x.livro]) + ' ' + str(x.capitulo) + ':' + str(x.versiculo), axis=1)
biblia_almeida = biblia_almeida.rename(columns={'texto': 'texto_almeida'})

biblia_almeida.set_index('ref', inplace=True)

biblia_almeida.drop(columns = ['cod', 'livro', 'capitulo', 'versiculo','testamento'], inplace=True)

print(biblia_almeida.shape)
print(biblia_almeida.columns)
biblia_almeida.sample(5)

(31482, 6)
Index(['texto_almeida', 'tempo', 'periodo', 'localizacao', 'autor',
       'tipo_livro'],
      dtype='object')


,texto_almeida,tempo,periodo,localizacao,autor,tipo_livro
ref,,,,,,
Dn 9:26,E depois das sessenta e duas semanas será cort...,-350,Persian,Babilônia,Daniel,Profetas Maiores
Sl 79:2,Deram os corpos mortos dos teus servos por com...,-550,Exilic,Israel,Davi,Poéticos
Mt 13:24,"Propôs-lhes outra parábola, dizendo: O reino d...",80,Christian,Israel,Mateus,Evangelho
At 15:32,"Depois Judas e Silas, que também eram profetas...",80,Christian,Roma,Lucas,Igreja Primitiva
Pv 2:15,Cujas veredas são tortuosas e que se desviam n...,-550,Exilic,Israel,Salomao,Poéticos


# BibleData

In [7]:
biblia_poliglota = pd.read_csv('/kaggle/input/datasets/bradystephenson/bibledata/AlamoPolyglot.csv')

In [8]:
print(biblia_poliglota.columns)
print(biblia_poliglota.book_name.drop_duplicates().to_list())

Index(['id', 'book_id', 'book_name', 'chapter', 'verse',
       'world_english_bible_web', 'king_james_bible_kjv', 'leningrad_codex',
       'jewish_publication_society_jps', 'codex_alexandrinus', 'brenton',
       'samaritan_pentateuch', 'samaritan_pentateuch_english',
       'onkelos_aramaic', 'onkelos_english'],
      dtype='object')
['Genesis', 'Exodus', 'Leviticus', 'Numbers', 'Deuteronomy', 'Joshua', 'Judges', 'Ruth', '1 Samuel', '2 Samuel', '1 Kings', '2 Kings', '1 Chronicles', '2 Chronicles', 'Ezra', 'Nehemiah', 'Esther', 'Job', 'Psalms', 'Proverbs', 'Ecclesiastes', 'Song of Solomon', 'Isaiah', 'Jeremiah', 'Lamentations', 'Ezekiel', 'Daniel', 'Hosea', 'Joel', 'Amos', 'Obadiah', 'Jonah', 'Micah', 'Nahum', 'Habakkuk', 'Zephaniah', 'Haggai', 'Zechariah', 'Malachi', 'Matthew', 'Mark', 'Luke', 'John', 'Acts', 'Romans', '1 Corinthians', '2 Corinthians', 'Galatians', 'Ephesians', 'Philippians', 'Colossians', '1 Thessalonians', '2 Thessalonians', '1 Timothy', '2 Timothy', 'Titus', 'Phi

In [9]:
siglas_en = { 'Haggai': 'Ag', 'Amos': 'Am', 'Revelation': 'Ap', 'Acts': 'At', 
             'Song of Solomon': 'Ct','Colossians': 'Cl', 
             'Daniel': 'Dn', 'Deuteronomy': 'Dt', 
             'Ecclesiastes': 'Ec', 'Ephesians': 'Ef','Ezra': 'Es', 'Esther': 'Et', 'Exodus': 'Ex', 'Ezekiel': 'Ez', 
             'Philemon': 'Flm', 'Philippians': 'Fl', 
             'Galatians': 'Gl', 'Genesis': 'Gn', 
             'Habakkuk':'Hc', 'Hebrews': 'Hb', 
             '1 Corinthians': '1Co', '1 Chronicles': '1Cr', '1 John': '1Jo', '1 Peter': '1Pe', '1 Kings': '1Re', 
             '1 Samuel': '1Sm', '1 Thessalonians': '1Ts', '1 Timothy': '1Tm', 
             '2 Corinthians': '2Co', '2 Chronicles': '2Cr', '2 John': '2Jo', '2 Peter': '2Pe', '2 Kings': '2Re', 
             '2 Samuel': '2Sm', '2 Thessalonians': '2Ts', '2 Timothy': '2Tm', 
             '3 John': '3Jo', 
             'Isaiah': 'Is', 
             'Jeremiah': 'Jr', 'Job': 'Jó', 'John': 'Jo', 'Joel': 'Jl', 'Jonah': 'Jon', 'Joshua': 'Js', 'Jude': 'Jd', 'Judges': 'Jz', 
             'Lamentations': 'Lm', 'Leviticus': 'Lv', 'Luke': 'Lc', 
             'Malachi': 'Ml', 'Mark': 'Mc', 'Matthew': 'Mt', 'Micah': 'Mq', 
             'Nahum': 'Na', 'Nehemiah': 'Ne', 'Numbers': 'Nm', 
             'Obadiah': 'Ob', 'Hosea': 'Os', 
             'Proverbs': 'Pv', 
             'Romans': 'Rm', 'Ruth': 'Rt', 
             'Psalms': 'Sl', 'Zephaniah': 'Sf', 
             'James': 'Tg', 'Titus': 'Tt', 
             'Zechariah': 'Zc' } 
print(len(siglas))

66


In [10]:
biblia_poliglota['ref'] = biblia_poliglota.apply(lambda x: str(siglas_en[x.book_name]) + ' ' + str(x.chapter) + ':' + str(x.verse), axis=1)
biblia_poliglota.set_index('ref', inplace=True)

biblia_poliglota.drop(columns = ['chapter', 'verse', 'id', 'book_id'], inplace=True)

print(biblia_poliglota.shape)
print(biblia_poliglota.columns)
biblia_poliglota.sample(5)


(31102, 11)
Index(['book_name', 'world_english_bible_web', 'king_james_bible_kjv',
       'leningrad_codex', 'jewish_publication_society_jps',
       'codex_alexandrinus', 'brenton', 'samaritan_pentateuch',
       'samaritan_pentateuch_english', 'onkelos_aramaic', 'onkelos_english'],
      dtype='object')


,book_name,world_english_bible_web,king_james_bible_kjv,leningrad_codex,jewish_publication_society_jps,codex_alexandrinus,brenton,samaritan_pentateuch,samaritan_pentateuch_english,onkelos_aramaic,onkelos_english
ref,,,,,,,,,,,
Jó 42:11,Job,"Then came there to him all his brothers, and a...","Then came there unto him all his brethren, and...",וַיָּבֹאוּ אֵלָיו כָּל־אֶחָיו וְכָל־*אחיתיו ...,"Then came there unto him all his brethren, and...",ἤκουσαν δὲ πάντες οἱ ἀδελφοὶ αὐτοῦ καὶ αἱ ἀδελ...,And all his brethren and his sisters heard all...,NaN,NaN,NaN,NaN
Lm 3:41,Lamentations,Let us lift up our heart with our hands to God...,Let us lift up our heart with our hands unto G...,נִשָּׂא לְבָבֵנוּ אֶל־כַּפָּיִם אֶל־אֵל בַּשׁ...,Let us lift up our heart with our hands Unto G...,ἐξηρευνήθη ἡ ὁδὸς ἡμῶν καὶ ἠτάσθη καὶ ἐπιστρέψ...,Let us lift up our hearts with our hand to the...,NaN,NaN,NaN,NaN
2Cr 34:30,2 Chronicles,"The king went up to the house of Yahweh, and a...",And the king went up into the house of the LOR...,וַיַּעַל הַמֶּלֶךְ בֵּית־יְהוָה וְכָל־אִישׁ ...,"And the king went up to the house of the LORD,...",καὶ ἀνέβη ὁ βασιλεὺς εἰς οἶκον κυρίου καὶ πᾶς ...,"And the king went up to the house of the Lord,...",NaN,NaN,NaN,NaN
Ap 9:6,Revelation,"In those days people will seek death, and will...","And in those days shall men seek death, and sh...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Lc 22:18,Luke,"for I tell you, I will not drink at all again ...","For I say unto you, I will not drink of the fr...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
livros_siglas = {
    # --- ANTIGO TESTAMENTO ---
    # Pentateuco
    "GEN": "Gn", "EXO": "Ex", "LEV": "Lv", "NUM": "Nm", "DEU": "Dt",
    # Históricos
    "JOS": "Js", "JDG": "Jz", "RUT": "Rt", "1SA": "1Sm", "2SA": "2Sm",
    "1KI": "1Re", "2KI": "2Re", "1CH": "1Cr", "2CH": "2Cr", "EZR": "Es",
    "NEH": "Ne", "EST": "Et",
    # Poéticos e Sabedoria
    "JOB": "Jó", "PSA": "Sl", "PRO": "Pv", "ECC": "Ec", "SNG": "Ct",
    # Profetas Maiores
    "ISA": "Is", "JER": "Jr", "LAM": "Lm", "EZK": "Ez", "DAN": "Dn",
    # Profetas Menores
    "HOS": "Os", "JOL": "Jl", "AMO": "Am", "OBA": "Ob", "JON": "Jon",
    "MIC": "Mq", "NAM": "Na", "HAB": "Hc", "ZEP": "Sf", "HAG": "Ag",
    "ZEC": "Zc", "MAL": "Ml",

    # --- NOVO TESTAMENTO ---
    # Evangelhos e Histórico
    "MAT": "Mt", "MRK": "Mc", "LUK": "Lc", "JHN": "Jo", "ACT": "At",
    # Cartas de Paulo
    "ROM": "Rm", "1CO": "1Co", "2CO": "2Co", "GAL": "Gl", "EPH": "Ef",
    "PHP": "Fl", "COL": "Cl", "1TH": "1Ts", "2TH": "2Ts", "1TM": "1Tm",
    "2TM": "2Tm", "TIT": "Tt", "PHM": "Flm",
    # Cartas Gerais e Apocalipse
    "HEB": "Hb", "JAS": "Tg", "1PE": "1Pe", "2PE": "2Pe", "1JN": "1Jo",
    "2JN": "2Jo", "3JN": "3Jo", "JUD": "Jd", "REV": "Ap"
}

biblia_strongs = pd.read_csv('/kaggle/input/datasets/bradystephenson/bibledata/HebrewStrongs.csv')
print(biblia_strongs.columns)
biblia_strongs.sample(5)

Index(['strongs_number', 'word', 'gloss', 'language', 'part_of_speech',
       'gender', 'occurrences', 'first_occurrence', 'root_word',
       'word_root_occurrence', 'first_root_number', 'first_root_hebrew',
       'second_root_number', 'second_root_hebrew', 'third_root_number',
       'third_root_hebrew'],
      dtype='object')


,strongs_number,word,gloss,language,part_of_speech,gender,occurrences,first_occurrence,root_word,word_root_occurrence,first_root_number,first_root_hebrew,second_root_number,second_root_hebrew,third_root_number,third_root_hebrew
6295,6296,פָּגַר,"pagar (paw-gar') v.\n1. to relax, i.e. become ...",H,verb,NaN,2,1SA 30:10,פגר,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3385,3386,יָרוַֹח,Yarowach (yaw-ro'-akh) n/p.\n1. (born at the) ...,H,noun proper,NaN,1,1CH 5:14,NaN,NaN,3394.0,ירח,NaN,NaN,NaN,NaN
3301,3302,יָפָה,yaphah (yaw-faw') v.\n1. (properly) to be brig...,H,verb,NaN,7,PSA 45:2,יפה,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6986,6987,קוֹטֶב,qoteb (ko'-teb) n-m.\n1. extermination\n[from ...,H,noun,masculine,1,HOS 13:14,NaN,NaN,6986.0,קטב,NaN,NaN,NaN,NaN
8554,8555,תִּמנָע,Timna` (tim-naw') n/p.\n1. restraint\n2. Timna...,H,noun proper,NaN,6,GEN 36:12,מנע,NaN,4513.0,מנע,NaN,NaN,NaN,NaN


In [12]:
# 1. Extrai a sigla em inglês (os 3 primeiros caracteres)
sigla_en = biblia_strongs['first_occurrence'].str[0:3]

# 2. Extrai o restante do texto (capítulo e versículo, a partir do caractere 3)
resto_ref = biblia_strongs['first_occurrence'].str[3:]

# 3. Traduz a sigla para PT e junta com o resto
biblia_strongs['ref'] = sigla_en.map(livros_siglas) + resto_ref

biblia_strongs.set_index('ref',inplace=True)

biblia_strongs

,strongs_number,word,gloss,language,part_of_speech,gender,occurrences,first_occurrence,root_word,word_root_occurrence,first_root_number,first_root_hebrew,second_root_number,second_root_hebrew,third_root_number,third_root_hebrew
ref,,,,,,,,,,,,,,,,
Gn 2:24,1,אָב,ab (awb) n-m.\n1. father\n{in a literal and im...,H,noun,masculine,1210,GEN 2:24,אב,1414.0,NaN,NaN,NaN,NaN,NaN,NaN
Ez 4:15,2,אַב,ab (ab) n-m.\n1. father\n[(Aramaic) correspond...,A,noun,masculine,9,EZK 4:15,אב,1414.0,1.0,אב,NaN,NaN,NaN,NaN
Jó 8:12,3,אֵב,'eb (abe) n-m.\n1. a green plant\n[from the s...,H,noun,masculine,2,JOB 8:12,אבב,13.0,24.0,אביב,NaN,NaN,NaN,NaN
Dn 4:12,4,אֵב,eb (abe) n-m.\n1. fruit\n[(Aramaic) correspond...,A,noun,masculine,3,DAN 4:12,אבב,13.0,3.0,אב,NaN,NaN,NaN,NaN
Et 1:10,5,אֲבַגתָּא,"Abagtha' (ab-ag-thaw') n/p.\n1. Abagtha, a eun...",H,noun proper,masculine,1,EST 1:10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1Sm 9:4,8670,תְּשׁוּרָה,tshuwrah (tesh-oo-raw') n-f.\n1. a gift\n[from...,H,noun,feminine,1,1SA 9:4,שׁיר שׁור,NaN,7788.0,שׁור,NaN,NaN,NaN,NaN
Lv 25:22,8671,תְּשִׁיעִי,tshiy`iy (tesh-ee-ee') adj.\n1. ninth\n[ord. f...,H,adjective,NaN,18,LEV 25:22,NaN,NaN,8672.0,תשׁע תשׁעה,NaN,NaN,NaN,NaN
Gn 5:5,8672,תֵּשַׁע תִּשׁעָה,tesha` (tay'-shah) (or (masculine) tishtah {ti...,H,noun enumeration,NaN,58,GEN 5:5,שׁעה,NaN,8159.0,שׁעה,NaN,NaN,NaN,NaN


# Concatenação

In [13]:
print(biblia_analisada.columns)
print(biblia_sentimentos.columns)
print(biblia_almeida.columns)

# 1. Mesclar tudo pela chave 'ref'
# Primeiro os textos pivotados...
biblia_final = pd.merge(biblia_analisada, biblia_sentimentos, on='ref', how='left')
# ... depois as entidades da biblia_analisada
biblia_final = pd.merge(biblia_final, biblia_almeida, on='ref', how='left')

biblia_final = biblia_final[['testamento', 'livro', 'capitulo', 'versiculo', 'texto_ACF',
           'texto_ARA', 'texto_ARIB', 'texto_NI', 'texto_NVI', 'texto_SBB', 'texto_almeida',
           'autor', 'tipo_livro','tempo', 'periodo', 'localizacao',
           'Pessoas', 'Locais', 'Organizacoes', 'Miscelanea',
           'label', 'score', 'posicao']]

# biblia_poliglota
biblia_final = pd.merge(biblia_final, biblia_poliglota, on='ref', how='left')

#biblia_strongs
biblia_final = pd.merge(biblia_final, biblia_strongs, on='ref', how='left')

# Visualizar o resultado limpo
print(biblia_final.shape)
print(biblia_final.columns)

Index(['testamento', 'livro', 'capitulo', 'versiculo', 'texto_ACF',
       'texto_ARA', 'texto_ARIB', 'texto_NI', 'texto_NVI', 'texto_SBB'],
      dtype='object')
Index(['posicao', 'texto', 'Pessoas', 'Locais', 'Organizacoes', 'Miscelanea',
       'label', 'score'],
      dtype='object')
Index(['texto_almeida', 'tempo', 'periodo', 'localizacao', 'autor',
       'tipo_livro'],
      dtype='object')
(34724, 50)
Index(['testamento', 'livro', 'capitulo', 'versiculo', 'texto_ACF',
       'texto_ARA', 'texto_ARIB', 'texto_NI', 'texto_NVI', 'texto_SBB',
       'texto_almeida', 'autor', 'tipo_livro', 'tempo', 'periodo',
       'localizacao', 'Pessoas', 'Locais', 'Organizacoes', 'Miscelanea',
       'label', 'score', 'posicao', 'book_name', 'world_english_bible_web',
       'king_james_bible_kjv', 'leningrad_codex',
       'jewish_publication_society_jps', 'codex_alexandrinus', 'brenton',
       'samaritan_pentateuch', 'samaritan_pentateuch_english',
       'onkelos_aramaic', 'onkelos_english',

In [14]:
biblia_final = biblia_final.replace({np.nan: None})


In [15]:
# Escolha aleatória
versiculo = biblia_final.iloc[random.randint(1,31102),:]

# .sample(1) pega 1 linha aleatória em qualquer tamanho de DataFrame
# .iloc[0] extrai essa linha como uma Série limpa
versiculo = biblia_final.sample(1).iloc[0]

print("-" * 50)
# Em uma Série, usamos .items() para pegar (nome_da_coluna, valor)
for col, valor in versiculo.items(): 
    if valor != None:
        print(f'{col:<15}: {valor}')
print("-" * 50)

--------------------------------------------------
testamento     : Novo Testamento
livro          : Mateus
capitulo       : 21
versiculo      : 26
texto_ACF      : E, se dissermos: Dos homens, tememos o povo, porque todos consideram João como profeta.
texto_ARA      : Mas, se dissermos: Dos homens, tememos o povo, porque todos consideram João como profeta.
texto_ARIB     : Mas, se dissermos: Dos homens, tememos o povo
texto_NI       : Mas, se dissermos: Dos homens, tememos o povo, porque todos consideram João como profeta.
texto_NVI      : Mas se dissermos: ‘dos homens’ — temos medo do povo, pois todos consideram João um profeta".
texto_SBB      : Mas se dissermos: Dos homens, tememos o povo
texto_almeida  : E, se dissermos: Dos homens, tememos o povo, porque todos consideram João como profeta.
autor          : Mateus
tipo_livro     : Evangelho
tempo          : 80
periodo        : Christian
localizacao    : Israel
Pessoas        : João
label          : negative
score          : 0.4786